In [ ]:
import json
import pickle
import pandas as pd

0: sample; 1: air

In [2]:
def get_dataframe(matrix, dir="plain", save_xlsx=False):
    with open(f"raw_data_{dir}/matrix_{matrix}/{matrix}.bmerawdata") as f:
        j = json.load(f)
    column_names = [el["name"] for el in j["rawDataBody"]["dataColumns"]]
    df = pd.DataFrame(j["rawDataBody"]["dataBlock"], columns=column_names)

    if save_xlsx:
        df.to_excel(f"raw_data_{dir}/matrix_{matrix}/{matrix}.xlsx")

    return df

In [3]:
# df = get_dataframe(1, save_xlsx=True)

In [4]:
def build_sensor_data(df):
    df["Date"] = pd.to_datetime(
        df["Real time clock"],
        unit="s",
        utc=True).map(lambda x: x.tz_convert("Europe/Istanbul"))
    df["Date"] = df["Date"].dt.tz_localize(None)

    sensor_indexes = sorted(df["Sensor Index"].unique())

    sensors = {}
    for i in sensor_indexes:
        sensors[i] = {}
        sensor = df[df["Sensor Index"] == i]
        heater_indexes = sorted(sensor["Heater Profile Step Index"].unique())
        for j in heater_indexes:
            sensors[i][j] = sensor[sensor["Heater Profile Step Index"] == j]

    return sensors

In [5]:
def build_raw_data_pickle(dir="plain", save_pickle=False):
    sensor_data = {}
    for matrix in range(2):
        df = get_dataframe(matrix)
        sensor_data[f"mat_{matrix}"] = build_sensor_data(df)

    if save_pickle:
        with open(f"raw_sensor_data_{dir}.pkl", "wb") as f:
            pickle.dump(sensor_data, f)

    return sensor_data

In [6]:
sensor_data = build_raw_data_pickle(dir="plain", save_pickle=True)

In [7]:
sensor_data["mat_0"][0][0]

,Sensor Index,Sensor ID,Time Since PowerOn,Real time clock,Temperature,Pressure,Relative Humidity,Resistance Gassensor,Heater Profile Step Index,Scanning Mode Enabled,Scanning Cycle Index,Label Tag,Error Code,Date
19,0,1607607109,15303,1699602881,20.092409,916.598022,26.403469,2.511035e+05,0,1,1,0,0,2023-11-10 10:54:41
140,0,1607607109,51659,1699602918,22.357891,916.564880,21.466579,7.183444e+05,0,1,1,0,0,2023-11-10 10:55:18
247,0,1607607109,88169,1699602954,23.827492,916.515930,19.904863,1.037225e+06,0,1,1,0,0,2023-11-10 10:55:54
367,0,1607607109,124632,1699602991,25.097506,916.466064,18.827410,1.297434e+06,0,1,1,0,0,2023-11-10 10:56:31
484,0,1607607109,161180,1699603027,26.242779,916.443970,17.727268,1.576294e+06,0,1,1,0,0,2023-11-10 10:57:07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95968,0,1607607109,30653330,1699633523,34.329979,913.740784,12.937240,6.339637e+07,0,1,1,0,0,2023-11-10 19:25:23
96082,0,1607607109,30689982,1699633560,34.359921,913.759033,13.036870,6.312160e+07,0,1,1,0,0,2023-11-10 19:26:00
96194,0,1607607109,30726576,1699633596,34.310013,913.806335,12.990450,6.316723e+07,0,1,1,0,0,2023-11-10 19:26:36
96312,0,1607607109,30763238,1699633633,34.305023,913.784668,13.009903,6.404691e+07,0,1,1,0,0,2023-11-10 19:27:13
